## HME_Matrices mit HME100K kombinieren

In [15]:
from pathlib import Path

hme_root = Path("/home/dedol002/.cache/kagglehub/datasets/cutedeadu/hme100k/versions/1")
mat_root = Path("HME_Matrices")

out_root = Path("hme100k_plus_matrices")
out_images = out_root / "images"
out_images.mkdir(parents=True, exist_ok=True)


In [2]:
from tqdm import tqdm
import shutil

hme_images = list((hme_root / "images").iterdir())

for img in tqdm(hme_images, desc="Copy HME images"):
    if img.is_file():
        target = out_images / img.name
        if not target.exists():
            shutil.copy2(img, target)

mat_images = list((mat_root / "images").iterdir())

for img in tqdm(mat_images, desc="Copy Matrix images"):
    if img.is_file():
        target = out_images / img.name
        if target.exists():
            raise RuntimeError(
                f"Dateiname-Kollision: {img.name} existiert schon in out/images"
            )
        shutil.copy2(img, target)

print("Bilder im Output:", len(list(out_images.iterdir())))


Copy Matrix images: 100%|██████████| 51/51 [00:00<00:00, 313.79it/s]


Bilder im Output: 99160


In [16]:
import re
from pathlib import Path

# Regex für chinesische Zeichen
chinese_symbols = re.compile(r'[\u4e00-\u9fff]')

hme_txt_path = hme_root / "train.txt"

with open(hme_txt_path, "r", encoding="utf-8") as f:
    hme_raw_lines = f.readlines()

hme_lines = []

for line in hme_raw_lines:
    line = line.rstrip("\n")

    # Zeilen mit chinesischen Zeichen überspringen
    if chinese_symbols.search(line):
        continue

    hme_lines.append(line)

mat_txt_path = mat_root / "train.txt"

with open(mat_txt_path, "r", encoding="utf-8") as f:
    mat_raw_lines = f.readlines()

mat_lines = []

for line in mat_raw_lines:
    line = line.rstrip("\n")

    if line.strip() == "":
        continue

    mat_lines.append(line)

print("HME lines (gefiltert):", len(hme_lines))
print("Matrix lines:", len(mat_lines))


HME lines (gefiltert): 99054
Matrix lines: 51


In [17]:
from sklearn.model_selection import train_test_split

mat_train, mat_val = train_test_split(mat_lines, test_size=0.2, random_state=42, shuffle=True)

hme_train, hme_temp = train_test_split(hme_lines, test_size=0.2, random_state=42, shuffle=True)
hme_val, hme_test = train_test_split(hme_temp, test_size=0.5, random_state=42, shuffle=True)

print("Matrizen train/val:", len(mat_train), len(mat_val))
print("HME train/val/test:", len(hme_train), len(hme_val), len(hme_test))


Matrizen train/val: 40 11
HME train/val/test: 79243 9905 9906


In [18]:
from pathlib import Path
import random

train_lines = []

# HME train
for line in hme_train:
    parts = line.split("\t", 1)
    img_path = parts[0].strip()
    label = parts[1].strip()
    img_name = Path(img_path).name

    new_line = "images/" + img_name + "\t" + label

    train_lines.append(new_line)

# Matrix train
for line in mat_train:
    parts = line.split("\t", 1)
    img_path = parts[0].strip()
    label = parts[1].strip()

    img_name = Path(img_path).name
    new_line = "images/" + img_name + "\t" + label

    train_lines.append(new_line)


val_lines = []

# HME val
for line in hme_val:
    parts = line.split("\t", 1)
    img_path = parts[0].strip()
    label = parts[1].strip()

    img_name = Path(img_path).name
    new_line = "images/" + img_name + "\t" + label

    val_lines.append(new_line)

# Matrix val
for line in mat_val:
    parts = line.split("\t", 1)
    img_path = parts[0].strip()
    label = parts[1].strip()

    img_name = Path(img_path).name
    new_line = "images/" + img_name + "\t" + label

    val_lines.append(new_line)


test_lines = []

# nur HME test (keine Matrizen)
for line in hme_test:
    parts = line.split("\t", 1)
    img_path = parts[0].strip()
    label = parts[1].strip()

    img_name = Path(img_path).name
    new_line = "images/" + img_name + "\t" + label

    test_lines.append(new_line)


In [ ]:
#ischen, damit Matrizen nicht alle am Ende stehen
import random
random.seed(42)
random.shuffle(train_lines)
random.shuffle(val_lines)
random.shuffle(test_lines)

In [19]:
from pathlib import Path

train_txt_path = out_root / "train.txt"

with open(train_txt_path, "w", encoding="utf-8") as f:
    for line in train_lines:
        f.write(line)
        f.write("\n")


val_txt_path = out_root / "val.txt"

with open(val_txt_path, "w", encoding="utf-8") as f:
    for line in val_lines:
        f.write(line)
        f.write("\n")


test_txt_path = out_root / "test.txt"

with open(test_txt_path, "w", encoding="utf-8") as f:
    for line in test_lines:
        f.write(line)
        f.write("\n")
